# SRA热插拔演示实验（多语言翻译插件）

在本笔记本中，我们演示了如何在不停止系统的情况下向正在运行的 SRA 模型动态添加新功能（新语言的 Synapse 翻译） - 热插拔。

### 实验场景
1. **模型 1**：在 `French -> English` 和 `German -> English` 上训练的基础模型。
2. **模型2**：仅在`Spanish -> English`上训练的扩展（插件）模型。
3. **添加**：将模型2的“西班牙语”参数注入到模型1中，并确认结果可以翻译所有三种语言。
4. **删除**：从模型1中分离“法语”参数，并确认不再只能翻译法语。

In [ ]:
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
try:
    from sra_language_models import MoESRALanguageModel
except ImportError:
    from src.sra_language_models import MoESRALanguageModel

torch.manual_seed(42)
random.seed(42)

## 1. 定义玩具数据集和词汇

In [ ]:
PARALLEL_DATA = [
    {"eng": "hello", "fra": "bonjour", "deu": "hallo", "spa": "hola"},
    {"eng": "apple", "fra": "pomme", "deu": "apfel", "spa": "manzana"},
    {"eng": "cat", "fra": "chat", "deu": "katze", "spa": "gato"},
    {"eng": "dog", "fra": "chien", "deu": "hund", "spa": "perro"},
    {"eng": "water", "fra": "eau", "deu": "wasser", "spa": "agua"},
    {"eng": "friend", "fra": "ami", "deu": "freund", "spa": "amigo"},
    {"eng": "thank you", "fra": "merci", "deu": "danke", "spa": "gracias"},
    {"eng": "goodbye", "fra": "au revoir", "deu": "auf wiedersehen", "spa": "adios"},
    {"eng": "house", "fra": "maison", "deu": "haus", "spa": "casa"},
    {"eng": "car", "fra": "voiture", "deu": "auto", "spa": "coche"}
]

vocab = ["[PAD]", "[ENG]", "[FRA]", "[DEU]", "[SPA]", "[SEP]", "[EOS]"]
for pair in PARALLEL_DATA:
    for word in pair.values():
        if word not in vocab:
            vocab.append(word)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

def encode(lang_tag, src_word, tgt_word=None):
    prompt = [word2idx[lang_tag], word2idx[src_word], word2idx["[SEP]"], word2idx["[ENG]"]]
    if tgt_word:
        target = [word2idx[tgt_word], word2idx["[EOS]"]]
        return prompt + target
    return prompt

print(f"Vocab Size: {VOCAB_SIZE}")


In [ ]:
from sra_reference import Router

def gumbel_router_forward(self, h, k_override=None):
    logits = torch.einsum("btd,nd->btn", h, self.synapse_emb) / self.scale
    if self.training:
        weights = F.gumbel_softmax(logits, tau=1.0, hard=True, dim=-1)
        idx = weights.argmax(dim=-1, keepdim=True)
        return idx, weights, logits
    else:
        vals, idx = torch.topk(logits, 1, dim=-1)
        weights = F.softmax(vals, dim=-1)
        return idx, weights, logits

Router.forward = gumbel_router_forward

def load_balance_loss(router_logits):
    if not router_logits: return torch.tensor(0.0)
    loss = 0.0
    for logits in router_logits:
        probs = F.softmax(logits, dim=-1)
        P = probs.mean(dim=(0, 1))
        idx = logits.argmax(dim=-1)
        f = torch.zeros_like(P)
        for i in range(logits.size(-1)):
            f[i] = (idx == i).float().mean()
        loss += (f * P).sum() * logits.size(-1)
    return loss

def get_batch(langs, batch_size=16):
    x = torch.zeros((batch_size, 6), dtype=torch.long)
    y = torch.full((batch_size, 6), -100, dtype=torch.long)
    for i in range(batch_size):
        lang = random.choice(langs)
        lang_tag = f"[{lang.upper()}]"
        pair = random.choice(PARALLEL_DATA)
        seq = encode(lang_tag, pair[lang], pair["eng"])
        x[i, :5] = torch.tensor(seq[:5])
        y[i, 3:5] = torch.tensor(seq[4:6])
    return x, y

def train_model(model, langs, steps=800, lr=2e-3):
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    model.train()
    for step in range(steps):
        x, y = get_batch(langs)
        logits, router_logits = model(x)
        ce_loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), ignore_index=-100)
        lb_loss = load_balance_loss(router_logits)
        loss = ce_loss + 0.5 * lb_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Trained on {langs} - Final CE Loss: {ce_loss.item():.4f}, LB Loss: {lb_loss.item():.4f}")

def test_model(model, langs_to_test):
    model.eval()
    with torch.no_grad():
        for lang in langs_to_test:
            lang_tag = f"[{lang.upper()}]"
            correct = 0
            for pair in PARALLEL_DATA:
                src_word = pair[lang]
                expected = pair["eng"]
                x = torch.tensor([encode(lang_tag, src_word)]).long()
                logits, router_logits = model(x)
                pred_idx = logits[0, 3].argmax().item()
                if idx2word[pred_idx] == expected:
                    correct += 1
            print(f"{lang.upper()} Accuracy: {correct}/{len(PARALLEL_DATA)}")
            if router_logits:
                x = torch.tensor([encode(lang_tag, PARALLEL_DATA[0][lang])]).long()
                _, router_logits = model(x)
                probs = F.softmax(router_logits[0][0, 3], dim=-1)
                top_synapse = probs.argmax().item()
                print(f"   [Routed to Synapse {top_synapse}] probs: {[round(p, 3) for p in probs.tolist()]}")


## 3.模型1和模型2独立训练
我们准备了两个基于 SRA 的小型语言模型并独立训练它们。
然而，**模型 1 中的词嵌入层 (Embedding) 和输出层 (Head) 是共享和冻结的**。
这再现了现实世界 LLM 插件开发中“训练附加模块，使其适合基本模型的向量空间 (API)”的标准实践。

In [ ]:
DIM = 64
model1 = MoESRALanguageModel(vocab_size=VOCAB_SIZE, dim=DIM, layers=1, num_synapses=2, k=1, syn_hidden=128)
model2 = MoESRALanguageModel(vocab_size=VOCAB_SIZE, dim=DIM, layers=1, num_synapses=1, k=1, syn_hidden=128)

print("Training Model 1 (fra, deu)...")
train_model(model1, ["fra", "deu"], steps=800)

# Model 2 is a plugin, so we make it conform to Model 1's API (shared layers)
model2.embed = model1.embed
model2.pos = model1.pos
model2.out = model1.out
model2.blocks[0].attn = model1.blocks[0].attn
model2.blocks[0].norm1 = model1.blocks[0].norm1
model2.blocks[0].norm2 = model1.blocks[0].norm2

# When training the plugin, freeze the shared layers so we don't damage the base model's knowledge
for name, param in model2.named_parameters():
    if not ("blocks.0.w" in name or "blocks.0.b" in name or "blocks.0.state" in name):
        param.requires_grad = False

print("\nTraining Model 2 (spa)...")
train_model(model2, ["spa"], steps=600)

print("\n--- Model 1 Test ---")
test_model(model1, ["fra", "deu", "spa"])


## 4. 实现热插拔（ADD）操作
对于 SRA 的矢量化 Expert (CausalMoESRABlock)，我们只需沿着张量的维度轴 (dim=0) 连接 (cat) 参数即可动态添加 Synapse。

In [ ]:
def hotswap_add_synapse(target_model, source_model):
    print("\n[Hot-Swap] Injecting the Spanish Synapse into the running Model 1 via tensor concatenation...")
    layer = target_model.blocks[0]
    src_layer = source_model.blocks[0]
    
    with torch.no_grad():
        layer.w1.data = torch.cat([layer.w1.data, src_layer.w1.data[0:1]], dim=0)
        layer.b1.data = torch.cat([layer.b1.data, src_layer.b1.data[0:1]], dim=0)
        layer.w2.data = torch.cat([layer.w2.data, src_layer.w2.data[0:1]], dim=0)
        layer.b2.data = torch.cat([layer.b2.data, src_layer.b2.data[0:1]], dim=0)
        layer.state.data = torch.cat([layer.state.data, src_layer.state.data[0:1]], dim=0)
        
        old_key = layer.router.synapse_emb.data
        src_key = src_layer.router.synapse_emb.data
        new_key = torch.cat([old_key, src_key[0:1]], dim=0)
        
        layer.router.synapse_emb = torch.nn.Parameter(new_key)
        layer.router.num_synapses += 1
        layer.num_synapses += 1
        
    print("Hot-Swap complete!")

def calibrate_router(model, langs, steps=400):
    print(f"\nRunning router calibration (re-training only the routing)...")
    for p in model.parameters(): p.requires_grad = False
    model.blocks[0].router.synapse_emb.requires_grad = True
    
    optimizer = torch.optim.AdamW([model.blocks[0].router.synapse_emb], lr=1e-2)
    model.train()
    for step in range(steps):
        x, y = get_batch(langs)
        logits, router_logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), ignore_index=-100)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    for p in model.parameters(): p.requires_grad = True
    print("Calibration complete!")

hotswap_add_synapse(model1, model2)
calibrate_router(model1, ["fra", "deu", "spa"], steps=500)

print("\n--- Model 1 Test (After Hot-Swap ADD & Calibration) ---")
test_model(model1, ["fra", "deu", "spa"])


## 5. 实现热插拔（DELETE/Lesion）操作
我们从模型 1 中剔除“French”突触参数。

In [ ]:
def hotswap_delete_synapse(target_model, idx_to_remove):
    print(f"\n[Hot-Swap] Removing Synapse {idx_to_remove} (French) from Model 1...")
    layer = target_model.blocks[0]
    
    with torch.no_grad():
        keep_indices = [i for i in range(layer.num_synapses) if i != idx_to_remove]
        
        layer.w1.data = layer.w1.data[keep_indices]
        layer.b1.data = layer.b1.data[keep_indices]
        layer.w2.data = layer.w2.data[keep_indices]
        layer.b2.data = layer.b2.data[keep_indices]
        layer.state.data = layer.state.data[keep_indices]
        
        new_key = layer.router.synapse_emb.data[keep_indices]
        layer.router.synapse_emb = nn.Parameter(new_key)
        
        layer.router.num_synapses -= 1
        layer.num_synapses -= 1
        
    print("Deletion complete!")

# Look up the routing target for French
model1.eval()
with torch.no_grad():
    x = torch.tensor([encode("[FRA]", "bonjour")]).long()
    _, router_logits = model1(x)
    fra_synapse_idx = router_logits[0][0, 3].argmax().item()
    
hotswap_delete_synapse(model1, fra_synapse_idx)

print("\n--- Model 1 Test (After Hot-Swap DELETE) ---")
test_model(model1, ["fra", "deu", "spa"])